# Mesh Tutorial 4: Neighbors, Adjacency, and Spatial Queries

This tutorial covers how to find neighbors and perform spatial queries:

1. **Topological Neighbors**: Neighbors based on mesh connectivity
2. **Adjacency Data Structures**: Efficient sparse encoding
3. **Spatial Queries with BVH**: Point containment and nearest-cell search
4. **Sampling**: Random points on cells, data interpolation

---

## Why This Matters for Physics-AI

- **Graph Neural Networks**: Need adjacency information for message passing
- **Data Augmentation**: Sample random points for training
- **Field Interpolation**: Query values at arbitrary locations
- **Collision Detection**: Find which cells contain query points

In [ ]:
import torch

from physicsnemo.mesh import Mesh
from physicsnemo.mesh.primitives.surfaces import sphere_icosahedral
from physicsnemo.mesh.primitives.planar import unit_square
from physicsnemo.mesh.primitives.volumes import cube_volume

## Section 1: Topological Neighbors

Topological neighbors are determined by mesh connectivity (which elements share vertices/edges),
not by spatial distance.

PhysicsNeMo-Mesh provides four adjacency queries:

| Method | Returns | Description |
|--------|---------|-------------|
| `get_point_to_points_adjacency()` | Points → Points | Graph edges (mesh skeleton) |
| `get_point_to_cells_adjacency()` | Points → Cells | Vertex star (cells containing each point) |
| `get_cell_to_cells_adjacency()` | Cells → Cells | Cells sharing a facet |
| `get_cells_to_points_adjacency()` | Cells → Points | Vertices of each cell |

### Point-to-Points (Graph Edges)

In [ ]:
mesh = sphere_icosahedral.load(subdivisions=1)
print(f"Mesh: {mesh.n_points} points, {mesh.n_cells} cells")

# Get adjacency: which points are connected to each point?
adj = mesh.get_point_to_points_adjacency()

# Convert to list-of-lists for inspection
neighbors_list = adj.to_list()

print(f"\nNeighbors of point 0: {neighbors_list[0]}")
print(f"Neighbors of point 1: {neighbors_list[1]}")
print(f"Neighbors of point 2: {neighbors_list[2]}")

In [ ]:
# Check vertex valence (number of neighbors)
valences = [len(n) for n in neighbors_list]
print(f"Valence distribution:")
for v in sorted(set(valences)):
    count = valences.count(v)
    print(f"  Valence {v}: {count} vertices")

### Point-to-Cells (Vertex Star)

In [ ]:
# Which cells contain each point?
adj_p2c = mesh.get_point_to_cells_adjacency()
cells_per_point = adj_p2c.to_list()

print(f"Cells containing point 0: {cells_per_point[0]}")
print(f"Cells containing point 1: {cells_per_point[1]}")

# Number of cells per vertex
n_cells_per_point = [len(c) for c in cells_per_point]
print(f"\nMean cells per vertex: {sum(n_cells_per_point) / len(n_cells_per_point):.1f}")

### Cell-to-Cells (Shared Facets)

In [ ]:
# Which cells share a facet (edge for triangles, face for tetrahedra)?
adj_c2c = mesh.get_cell_to_cells_adjacency(adjacency_codimension=1)
cell_neighbors = adj_c2c.to_list()

print(f"Neighbors of cell 0: {cell_neighbors[0]}")
print(f"Neighbors of cell 1: {cell_neighbors[1]}")

# For triangles, each cell has up to 3 neighbors (one per edge)
n_neighbors = [len(n) for n in cell_neighbors]
print(f"\nNeighbor count distribution:")
for n in sorted(set(n_neighbors)):
    count = n_neighbors.count(n)
    print(f"  {n} neighbors: {count} cells")

### Cells-to-Points (Cell Vertices)

In [ ]:
# Which points define each cell?
adj_c2p = mesh.get_cells_to_points_adjacency()
cell_vertices = adj_c2p.to_list()

print(f"Vertices of cell 0: {cell_vertices[0]}")
print(f"Vertices of cell 1: {cell_vertices[1]}")

# This is essentially the cells tensor in list form
print(f"\nCompare to mesh.cells[0]: {mesh.cells[0].tolist()}")

## Section 2: Adjacency Data Structure

Internally, PhysicsNeMo-Mesh uses an efficient sparse encoding:

- **indices**: Flat array of all neighbor indices
- **offsets**: Start position in `indices` for each element

This is the same format used by PyTorch Geometric (CSR-style).

In [ ]:
mesh = sphere_icosahedral.load(subdivisions=2)
adj = mesh.get_point_to_points_adjacency()

print(f"Adjacency object: {adj}")
print(f"\nIndices shape: {adj.indices.shape}")
print(f"Offsets shape: {adj.offsets.shape}")

In [ ]:
# How to read the sparse format:
# Neighbors of point i are: indices[offsets[i]:offsets[i+1]]

i = 5  # Example point
start = adj.offsets[i].item()
end = adj.offsets[i + 1].item()
neighbors = adj.indices[start:end]

print(f"Point {i} neighbors (sparse): {neighbors.tolist()}")
print(f"Point {i} neighbors (list):   {adj.to_list()[i]}")

### Converting to PyTorch Geometric Format

For GNN libraries, you often need edge indices in COO format.

In [ ]:
# Convert adjacency to edge_index (COO format)
# edge_index[0] = source nodes, edge_index[1] = target nodes

adj = mesh.get_point_to_points_adjacency()

# Build source indices by repeating each point index by its neighbor count
neighbor_counts = adj.offsets[1:] - adj.offsets[:-1]
source = torch.repeat_interleave(torch.arange(mesh.n_points), neighbor_counts)
target = adj.indices

edge_index = torch.stack([source, target], dim=0)
print(f"edge_index shape: {edge_index.shape}")
print(f"Number of edges: {edge_index.shape[1]}")
print(f"\nFirst 10 edges:")
print(edge_index[:, :10])

## Section 3: Spatial Queries with BVH

For spatial queries (which cells contain a point? what's the nearest cell?),
PhysicsNeMo-Mesh provides a Bounding Volume Hierarchy (BVH).

BVH enables O(log N) query time instead of O(N) brute-force.

In [ ]:
from physicsnemo.mesh.spatial import BVH

# Create a 2D mesh for clear visualization
mesh = unit_square.load(subdivisions=4)
print(f"Mesh: {mesh.n_cells} cells")

# Build BVH
bvh = BVH.from_mesh(mesh)
print(f"\nBVH: {bvh.n_nodes} nodes")

In [ ]:
# Query: which cells might contain these points?
query_points = torch.tensor([
    [0.25, 0.25],  # Inside mesh
    [0.5, 0.5],    # Center
    [0.75, 0.75],  # Inside mesh
    [1.5, 1.5],    # Outside mesh
])

candidates = bvh.find_candidate_cells(query_points)

for i, (pt, cands) in enumerate(zip(query_points, candidates)):
    print(f"Point {pt.tolist()}: {len(cands)} candidate cells")

### Point Containment

Find which cell(s) actually contain each query point.

In [ ]:
from physicsnemo.mesh.sampling.sample_data import find_containing_cells

# Find containing cells (returns first containing cell for each point)
cell_indices, bary_coords = find_containing_cells(mesh, query_points)

print("Point containment results:")
for i, (pt, cell_idx) in enumerate(zip(query_points, cell_indices)):
    if cell_idx >= 0:
        print(f"  {pt.tolist()} -> cell {cell_idx.item()}")
    else:
        print(f"  {pt.tolist()} -> outside mesh")

In [ ]:
# The barycentric coordinates tell you where in the cell the point is
print("\nBarycentric coordinates:")
for i, (pt, bary) in enumerate(zip(query_points, bary_coords)):
    if not bary.isnan().any():
        print(f"  {pt.tolist()}: {bary.tolist()}")
    else:
        print(f"  {pt.tolist()}: outside (NaN)")

## Section 4: Sampling Points on Meshes

PhysicsNeMo-Mesh can sample random points on mesh cells and interpolate data at those points.

### Random Point Sampling

In [ ]:
mesh = sphere_icosahedral.load(subdivisions=2)

# Sample one random point per cell (default)
random_points = mesh.sample_random_points_on_cells()
print(f"Random points shape: {random_points.shape}")
print(f"  (one point per cell, in 3D space)")

In [ ]:
# Sample multiple points from specific cells
# Sample 5 points from cell 0, 3 points from cell 1, 2 points from cell 2
cell_indices = torch.tensor([0, 0, 0, 0, 0, 1, 1, 1, 2, 2])
random_points = mesh.sample_random_points_on_cells(cell_indices=cell_indices)

print(f"Sampled {len(random_points)} points")
print(f"Points from cell 0:\n{random_points[:5]}")

In [ ]:
# Control the distribution with alpha parameter
# alpha=1.0 (default): uniform over the cell
# alpha>1: concentrated toward center
# alpha<1: concentrated toward edges/vertices

cell_indices = torch.zeros(100, dtype=torch.long)  # Sample 100 points from cell 0

uniform = mesh.sample_random_points_on_cells(cell_indices=cell_indices, alpha=1.0)
centered = mesh.sample_random_points_on_cells(cell_indices=cell_indices, alpha=5.0)

print(f"Uniform sampling std: {uniform.std(dim=0)}")
print(f"Centered sampling std: {centered.std(dim=0)}")

### Data Interpolation at Query Points

In [ ]:
# Create a mesh with data
mesh = unit_square.load(subdivisions=4)
mesh.point_data["temperature"] = mesh.points[:, 0] + mesh.points[:, 1]  # T = x + y
mesh.cell_data["pressure"] = torch.randn(mesh.n_cells)

# Query points inside the mesh
query_points = torch.tensor([
    [0.25, 0.25],
    [0.5, 0.5],
    [0.75, 0.25],
])

# Sample point data (interpolated using barycentric coordinates)
sampled_point_data = mesh.sample_data_at_points(query_points, data_source="points")
print("Interpolated point data:")
print(f"  Temperature at query points: {sampled_point_data['temperature']}")
print(f"  Expected (x + y): {query_points.sum(dim=-1)}")

In [ ]:
# Sample cell data (no interpolation, just cell value)
sampled_cell_data = mesh.sample_data_at_points(query_points, data_source="cells")
print("Cell data at query points:")
print(f"  Pressure: {sampled_cell_data['pressure']}")

### Handling Points Outside the Mesh

In [ ]:
# Query points including some outside
query_points = torch.tensor([
    [0.5, 0.5],   # Inside
    [1.5, 0.5],   # Outside
    [-0.1, 0.5],  # Outside
])

# Default behavior: NaN for points outside
sampled = mesh.sample_data_at_points(query_points, data_source="points")
print(f"Temperature (with NaN for outside): {sampled['temperature']}")

In [ ]:
# Alternative: project to nearest cell first
sampled_projected = mesh.sample_data_at_points(
    query_points, 
    data_source="points",
    project_onto_nearest_cell=True
)
print(f"Temperature (projected): {sampled_projected['temperature']}")

## Section 5: Using Neighbors for Message Passing

Here's how you might use adjacency information for GNN-style operations.

In [ ]:
mesh = sphere_icosahedral.load(subdivisions=2)

# Create some node features
features = torch.randn(mesh.n_points, 8)

# Get adjacency
adj = mesh.get_point_to_points_adjacency()

# Simple message passing: average neighbor features
# This is the core of many GNN architectures

# Build edge_index
neighbor_counts = adj.offsets[1:] - adj.offsets[:-1]
source = torch.repeat_interleave(torch.arange(mesh.n_points), neighbor_counts)
target = adj.indices

# Gather neighbor features
neighbor_features = features[target]  # (n_edges, n_features)

# Aggregate by source node (mean pooling)
from physicsnemo.mesh.utilities._scatter_ops import scatter_aggregate

aggregated = scatter_aggregate(
    src_data=neighbor_features,
    src_to_dst_mapping=source,
    n_dst=mesh.n_points,
    aggregation="mean",
)

print(f"Original features shape: {features.shape}")
print(f"Aggregated features shape: {aggregated.shape}")

## Summary

In this tutorial, you learned about mesh queries:

1. **Topological Neighbors**:
   - `get_point_to_points_adjacency()` - graph edges
   - `get_point_to_cells_adjacency()` - vertex star
   - `get_cell_to_cells_adjacency()` - cell neighbors

2. **Adjacency Format**: Sparse `(indices, offsets)` encoding, convertible to COO

3. **Spatial Queries**:
   - `BVH.from_mesh()` - build acceleration structure
   - `find_containing_cells()` - point-in-cell test

4. **Sampling**:
   - `sample_random_points_on_cells()` - random point generation
   - `sample_data_at_points()` - data interpolation

---

### Next Steps

- **Tutorial 5: Quality & Repair** - Mesh validation and repair
- **Tutorial 6: ML Integration** - Performance, datapipes, torch.compile